In [1]:
from langchain_community.document_loaders import HuggingFaceDatasetLoader 

dataset_name = "databricks/databricks-dolly-15k"
page_content_column = "context"

In [2]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

# Load environment variables from .env file
load_dotenv()

# Get HF token from environment variable
hf_token = os.getenv("HF_TOKEN")

# Authenticate with HuggingFace using your token
login(token=hf_token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
import pprint
loader = HuggingFaceDatasetLoader(dataset_name, page_content_column)
data = loader.load()
pprint.pp(data[:2]) 

[Document(metadata={'instruction': 'When did Virgin Australia start operating?', 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}, page_content='"Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia\'s domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney."'),
 Document(metadata={'instruction': 'Which is a species of fish? Tope or Rope', 'response': 'Tope', 'category': 'classification'}, page_content='""')]


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

# Split the loaded documents:
docs = text_splitter.split_documents(data)
print(docs[0]) # Optional: Print the first document chunk

page_content='"Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney."' metadata={'instruction': 'When did Virgin Australia start operating?', 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}


In [5]:
from langchain_community.embeddings import HuggingFaceEmbeddings

modelPath = "sentence-transformers/all-MiniLM-l6-v2"
model_kwargs = {'device':'cpu'}
encode_kwargs = {'normalize_embeddings': False}

embeddings = HuggingFaceEmbeddings(
  model_name=modelPath,
  model_kwargs=model_kwargs,
  encode_kwargs=encode_kwargs
)

/var/folders/8g/b0fs32r14l57mfjp3l9lh_nw0000gn/T/ipykernel_15630/3707483576.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
text = "This is a test document."
query_result = embeddings.embed_query(text)
print(query_result[:3])

[-0.038338519632816315, 0.12346471101045609, -0.028642937541007996]


In [7]:
from langchain_community.vectorstores import FAISS

db = FAISS.from_documents(docs, embeddings)

In [8]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline
from langchain_community.llms import HuggingFacePipeline

In [9]:
tokenizer = AutoTokenizer.from_pretrained("Intel/dynamic_tinybert")
model = AutoModelForQuestionAnswering.from_pretrained("Intel/dynamic_tinybert")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: Intel/dynamic_tinybert
Key              | Status     |  | 
-----------------+------------+--+-
fit_dense.bias   | UNEXPECTED |  | 
fit_dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Note about changes from the original exercise code

The original exercise asks us to create a Hugging Face `"question-answering"` pipeline with:

```python
pipeline(
    "question-answering",
    model="Intel/dynamic_tinybert",
    tokenizer=tokenizer,
    return_tensors="pt"
)
```

However, in my installed version of transformers, "question-answering" is not available as a registered pipeline task. Running the original code raises:

```
KeyError: "Unknown task question-answering"
```
The error message lists the pipeline tasks available in this environment, and "question-answering" is not included. I also tried the older "text2text-generation" task, but that task is also unavailable in this environment.

Because the exercise later wraps the pipeline with LangChain’s HuggingFacePipeline, the replacement pipeline needs to be compatible with LangChain’s text-in/text-out LLM interface. Therefore, I replaced the unavailable "question-answering" pipeline with the available "text-generation" pipeline.

This keeps the intended RAG structure of the exercise:

1. load documents from the Hugging Face dataset
2. split the documents into chunks
3. create embeddings
4. store them in FAISS
5. retrieve relevant chunks
6. pass the retrieved context to a Hugging Face model through LangChain
7. generate an answer

The original Intel/dynamic_tinybert model is an extractive question-answering model, which predicts start and end token positions inside a context. The replacement text-generation model instead receives the retrieved context and question inside a prompt and generates an answer. This is not exactly the same internal mechanism, but it preserves the goal of the exercise: implementing a RAG pipeline with LangChain, Hugging Face, embeddings, FAISS retrieval, and a Hugging Face model wrapped by HuggingFacePipeline.

I also updated the RetrievalQA import because newer LangChain versions moved legacy chains out of langchain.chains. In this environment, the compatible import is:

```
from langchain_classic.chains import RetrievalQA
```

instead of:

```
from langchain.chains import RetrievalQA
```

Finally, I used .invoke({"query": question}) instead of .run(...) because newer LangChain versions prefer the .invoke() interface.

In [14]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_community.llms import HuggingFacePipeline

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

Youtubeer = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=250,
    min_new_tokens=30,
    do_sample=False,
    return_full_text=False,
    clean_up_tokenization_spaces=False,
    pad_token_id=tokenizer.eos_token_id,
    repetition_penalty=1.1
)

llm = HuggingFacePipeline(
    pipeline=Youtubeer
)

llm = HuggingFacePipeline(
    pipeline=Youtubeer,
    model_kwargs={"temperature": 0.7}
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'min_new_tokens', 'do_sample', 'repetition_penalty', 'max_new_tokens', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [11]:
%pip install -q langchain-classic

from langchain_classic.chains import RetrievalQA

retriever = db.as_retriever(search_kwargs={"k": 4})

qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=False
)

Note: you may need to restart the kernel to use updated packages.


In [15]:
question = "What is cheesemaking?"

result = qa.run(question)

print(result)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 Cheesemaking is the process of making cheese. It involves bringing milk (or milk from a cow, goat, sheep, or buffalo) to a temperature required to promote the growth of bacteria that feed on lactose and thus ferment the lactose into lactic acid. The bacteria in the milk may be wild, as is the case with unpasteurised milk, added from a culture, frozen or freeze dried concentrate of starter bacteria. Bacteria which produce only lactic acid during fermentation are homofermentative; those that also produce lactic acid and other compounds such as carbon dioxide, alcohol, aldehy
